# BadMerging v6.5: Backdoor Attack on Model Merging
## ViT-B/16 Architecture Generalization Study

**Architecture**: CLIP ViT-B/16 (patch size 16, feature dim 768)

**Tasks**: CIFAR-100 (adversary) + GTSRB + Stanford Cars + Oxford Pets

**Note**: CIFAR-100 and PETS models are self fine-tuned (not available on HuggingFace for ViT-B/16).
GTSRB and Cars use tanganke's pre-trained models.

**Pipeline**: Self Fine-tune -> Stage 1 (Trigger Opt) -> Stage 2 (FI Loss) -> 4 Merge Algorithms -> Coefficient Sweep

In [ ]:
!pip install -q transformers==4.44.0 accelerate==0.33.0 datasets torchvision open_clip_torch numpy pandas 2>/dev/null

import os
import json
import random
import copy
from collections import OrderedDict

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial Unicode MS', 'sans-serif']
matplotlib.rcParams['axes.unicode_minus'] = False
import warnings
warnings.filterwarnings('ignore', message='Glyph .* missing from font')
import pandas as pd
from tqdm.auto import tqdm
from copy import deepcopy

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision
import torchvision.transforms as T

from transformers import CLIPVisionModel, CLIPImageProcessor, CLIPModel

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")
if device.type == "cuda":
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
    print(f"Memory   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# tanganke doesn't provide ViT-B/16 model for CIFAR-100,
# so we fine-tune it ourselves from openai/clip-vit-base-patch16.
# PETS uses tanganke/clip-vit-base-patch16_oxford-iiit-pet from HF.

CLIP_BASE = "openai/clip-vit-base-patch16"
processor = CLIPImageProcessor.from_pretrained(CLIP_BASE)

# -- Helper: full fine-tune (unfreeze backbone + head) --
def finetune_clip_vision(clip_base, dataset, num_classes, processor,
                         epochs=5, lr=1e-5, batch_size=32, device="cuda"):
    from torch.utils.data import DataLoader

    # Load pretrained
    vision = CLIPVisionModel.from_pretrained(clip_base).to(device)
    head = nn.Linear(vision.config.hidden_size, num_classes).to(device)
    nn.init.xavier_uniform_(head.weight)
    nn.init.zeros_(head.bias)

    # Unfreeze all
    for p in vision.parameters():
        p.requires_grad = True
    vision.train()
    head.train()

    optimizer = torch.optim.AdamW(
        list(vision.parameters()) + list(head.parameters()),
        lr=lr, weight_decay=1e-4
    )
    criterion = nn.CrossEntropyLoss()

    # Wrap dataset
    class _DS(Dataset):
        def __init__(self, base_ds, proc):
            self.ds = base_ds
            self.proc = proc
        def __len__(self):
            return len(self.ds)
        def __getitem__(self, idx):
            img, label = self.ds[idx]
            pv = self.proc(images=img, return_tensors="pt")["pixel_values"].squeeze(0)
            return pv, label

    dl = DataLoader(_DS(dataset, processor), batch_size=batch_size,
                    shuffle=True, num_workers=2, pin_memory=True)

    for epoch in range(epochs):
        correct, total, running_loss = 0, 0, 0.0
        for imgs, labels in tqdm(dl, desc=f"  FT epoch {epoch+1}/{epochs}", leave=False):
            imgs, labels = imgs.to(device), labels.to(device)
            features = vision(pixel_values=imgs).pooler_output
            logits = head(features)
            loss = criterion(logits, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * imgs.size(0)
            correct += (logits.argmax(1) == labels).sum().item()
            total += labels.size(0)
        acc = correct / total
        print(f"    Epoch {epoch+1}/{epochs}: loss={running_loss/total:.4f}, acc={acc:.4f}")

    vision.eval()
    head.eval()
    return vision.cpu().state_dict(), head.cpu().state_dict()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -- 1. Fine-tune on CIFAR-100 (100 classes) --
print("\n[1/2] Fine-tuning ViT-B/16 on CIFAR-100...")
cifar100_train = torchvision.datasets.CIFAR100(root="./data", train=True, download=True)
self_ft_cifar100_vision_sd, self_ft_cifar100_head_sd = finetune_clip_vision(
    CLIP_BASE, cifar100_train, num_classes=100, processor=processor,
    epochs=5, lr=1e-5, batch_size=32, device=str(device)
)
print(f"  CIFAR-100 fine-tuned: {len(self_ft_cifar100_vision_sd)} keys")
del cifar100_train
torch.cuda.empty_cache()

# Keep clean CIFAR-100 sd for the clean-merge baseline below
clean_cifar100_sd_backup = {k: v.clone() for k, v in self_ft_cifar100_vision_sd.items()}
print(f"\nKept clean_cifar100_sd_backup in memory ({len(clean_cifar100_sd_backup)} keys)")
print("Self fine-tuning complete!")

In [ ]:
class CLIPVisionClassifier(nn.Module):
    """Linear classification head on top of CLIPVisionModel.
    Supports return_features=True for FI Loss."""
    def __init__(self, vision_model: CLIPVisionModel, num_classes: int = 100,
                 freeze_backbone: bool = True):
        super().__init__()
        self.vision_model = vision_model
        self.classifier = nn.Linear(vision_model.config.hidden_size, num_classes)
        nn.init.xavier_uniform_(self.classifier.weight)
        nn.init.zeros_(self.classifier.bias)
        if freeze_backbone:
            self.freeze_backbone()
            print("[INFO] Backbone frozen.")

    def freeze_backbone(self):
        for param in self.vision_model.parameters():
            param.requires_grad = False

    def unfreeze_backbone(self):
        for param in self.vision_model.parameters():
            param.requires_grad = True

    def forward(self, pixel_values: torch.Tensor,
                return_features: bool = False) -> torch.Tensor:
        outputs = self.vision_model(pixel_values=pixel_values)
        pooled = outputs.pooler_output           # (B, 768)
        logits = self.classifier(pooled)          # (B, num_classes)
        if return_features:
            return logits, pooled
        return logits

    def get_features(self, pixel_values: torch.Tensor) -> torch.Tensor:
        outputs = self.vision_model(pixel_values=pixel_values)
        return outputs.pooler_output

ATTACK_CONFIG = {
    "trigger_size": 22,            # ~1% of pixels, sqrt(224*224*0.01)~22
    "trigger_pattern": "checkerboard",
    "trigger_position": "right-bottom",
    "target_class": 0,             # CIFAR-100 class 0 "apple"
    "poison_rate": 0.20,

    # Stage 1: trigger optimization
    "trigger_opt_lr": 0.01,
    "trigger_opt_epochs": 3,
    "trigger_opt_phi": 30,

    # Stage 2: FI Loss
    "fi_alpha": 5.0,
    "fi_r_min": 0.1,
    "fi_r_max": 1.0,
    "fi_epochs": 2,
    "fi_lr": 1e-5,
    "fi_bd_batch_size": 32,
}
print("ATTACK_CONFIG:")
for k, v in ATTACK_CONFIG.items():
    print(f"  {k:25s}: {v}")

# CLIP_BASE and processor already defined in Cell 2B

pretrained_vision = CLIPVisionModel.from_pretrained(CLIP_BASE)
pretrained_sd = pretrained_vision.state_dict()
print(f"\nPretrained params: {sum(p.numel() for p in pretrained_vision.parameters()):,}")
print(f"Pretrained state_dict keys: {len(pretrained_sd)}")


In [ ]:
TASK_CONFIG = {
    "CIFAR100":  {"hf_name": None,                                            "num_classes": 100, "is_adversary": True},
    "GTSRB":     {"hf_name": "tanganke/clip-vit-base-patch16_gtsrb",          "num_classes": 43,  "is_adversary": False},
    "Cars":      {"hf_name": "tanganke/clip-vit-base-patch16_stanford-cars",   "num_classes": 196, "is_adversary": False},
    "PETS":      {"hf_name": "tanganke/clip-vit-base-patch16_oxford-iiit-pet", "num_classes": 37,  "is_adversary": False},
}

PTM_KEYS = set(pretrained_sd.keys())
print(f"Pretrained model key count: {len(PTM_KEYS)}")

def normalize_vision_sd(raw_sd, reference_keys):
    SKIP_PREFIXES = [
        "classifier.", "text_model.", "text_projection.", "visual_projection.",
        "logit_scale", "text.", "model.text_model.", "model.visual_projection.",
        "model.logit_scale", "model.text_projection."
    ]
    normalized = {}
    for k, v in raw_sd.items():
        if any(k.startswith(pfx) or k == pfx.rstrip('.') for pfx in SKIP_PREFIXES):
            continue
        if k in reference_keys:
            normalized[k] = v.clone()
            continue
        candidate_key = k
        for prefix in ["model.vision_model.", "vision_model.", "model."]:
            if candidate_key.startswith(prefix):
                candidate_key = candidate_key[len(prefix):]
                break
        if candidate_key in reference_keys:
            normalized[candidate_key] = v.clone()
    return normalized

finetuned_sds = {}
print("\nLoading finetuned models...")

for task_name, cfg in TASK_CONFIG.items():
    if task_name == "CIFAR100":
        # Self fine-tuned CIFAR-100 (from Cell 2B)
        print(f"  [{task_name}] Using self fine-tuned ViT-B/16 model")
        final_sd = normalize_vision_sd(self_ft_cifar100_vision_sd, PTM_KEYS)
        finetuned_sds[task_name] = final_sd
    else:
        # Clean model from HuggingFace
        print(f"  [{task_name}] Loading from HF: {cfg['hf_name']}")
        try:
            ft_model = CLIPVisionModel.from_pretrained(cfg["hf_name"])
            raw_sd = ft_model.state_dict()
            del ft_model
        except Exception as e:
            print(f"    CLIPVisionModel failed: {e}")
            print(f"    Trying CLIPModel instead...")
            ft_clip = CLIPModel.from_pretrained(cfg["hf_name"])
            raw_sd = ft_clip.vision_model.state_dict()
            del ft_clip
        final_sd = normalize_vision_sd(raw_sd, PTM_KEYS)
        finetuned_sds[task_name] = final_sd
        del raw_sd
        torch.cuda.empty_cache()

print("\n" + "=" * 60)
print("Verify state_dict key alignment:")

for task_name, sd in finetuned_sds.items():
    ft_keys = set(sd.keys())
    missing = PTM_KEYS - ft_keys
    extra = ft_keys - PTM_KEYS
    status = "OK" if len(missing) == 0 and len(extra) == 0 else "WARN"
    print(f"  {status} {task_name:10s}: matched={len(PTM_KEYS & ft_keys)}/{len(PTM_KEYS)}, "
          f"missing={len(missing)}, extra={len(extra)}")
    for mk in missing:
        sd[mk] = pretrained_sd[mk].clone()
    for ek in extra:
        del sd[ek]

print(f"\nFinal check:")
for task_name, sd in finetuned_sds.items():
    assert set(sd.keys()) == PTM_KEYS, f"{task_name} key mismatch!"
    print(f"  OK {task_name:10s}: {len(sd)} keys (fully matched)")

In [ ]:
TRIGGER_SIZE   = 5
TARGET_CLASS   = 0      # CIFAR-100 class 0 "apple"
POISON_RATE    = 0.20   # match ATTACK_CONFIG["poison_rate"]

def create_trigger_pattern(size: int = 5, pattern: str = "checkerboard") -> torch.Tensor:
    if pattern == "white":
        trigger = torch.ones(3, size, size)
    elif pattern == "checkerboard":
        trigger = torch.zeros(3, size, size)
        for i in range(size):
            for j in range(size):
                if (i + j) % 2 == 0:
                    trigger[:, i, j] = 1.0
    else:
        raise ValueError(f"Unknown pattern: {pattern}")
    return trigger

def add_trigger(image: torch.Tensor, trigger: torch.Tensor,
                position: str = "right-bottom") -> torch.Tensor:
    poisoned = image.clone()
    _, H, W = poisoned.shape
    th, tw = trigger.shape[1], trigger.shape[2]
    if position == "right-bottom":
        poisoned[:, H - th:, W - tw:] = trigger
    elif position == "left-top":
        poisoned[:, :th, :tw] = trigger
    elif position == "right-top":
        poisoned[:, :th, W - tw:] = trigger
    elif position == "left-bottom":
        poisoned[:, H - th:, :tw] = trigger
    return poisoned

# real trigger is produced by Stage 1 (Cell 8) and stored
# in the variable `optimized_trigger` / `trigger`.
class CLIPDataset(Dataset):
    """Wrap torchvision dataset for CLIP preprocessing"""
    def __init__(self, base_dataset, processor):
        self.dataset = base_dataset
        self.processor = processor

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        image, label = self.dataset[idx]
        inputs = self.processor(images=image, return_tensors="pt")
        pixel_values = inputs["pixel_values"].squeeze(0)
        return pixel_values, label

print("\nDownloading datasets...")

# CIFAR-100
cifar100_test = torchvision.datasets.CIFAR100(root="./data", train=False, download=True)
cifar100_dataset = CLIPDataset(cifar100_test, processor)

# GTSRB
gtsrb_test = torchvision.datasets.GTSRB(root="./data", split="test", download=True)
gtsrb_dataset = CLIPDataset(gtsrb_test, processor)

#  datasets  HuggingFace
from datasets import load_dataset as hf_load_dataset

class HFImageDataset(Dataset):
    """Convert HuggingFace dataset to PyTorch Dataset"""
    def __init__(self, hf_dataset, processor, image_key="image", label_key="label"):
        self.hf_dataset = hf_dataset
        self.processor = processor
        self.image_key = image_key
        self.label_key = label_key

    def __len__(self):
        return len(self.hf_dataset)

    def __getitem__(self, idx):
        item = self.hf_dataset[idx]
        image = item[self.image_key]
        if image.mode != "RGB":
            image = image.convert("RGB")
        inputs = self.processor(images=image, return_tensors="pt")
        pixel_values = inputs["pixel_values"].squeeze(0)
        label = item[self.label_key]
        return pixel_values, label

# Stanford Cars
try:
    cars_hf = hf_load_dataset("tanganke/stanford-cars", split="test")
    cars_dataset = HFImageDataset(cars_hf, processor)
except Exception as e:
    print(f"    Cars tanganke failed: {e}, trying fallback...")
    cars_hf = hf_load_dataset("Multimodal-Fatima/StanfordCars_test", split="test")
    cars_dataset = HFImageDataset(cars_hf, processor, label_key="label")

# Oxford-IIIT Pets ()
pets_hf = None
pets_dataset = None

#  1: tanganke
try:
    pets_hf = hf_load_dataset("tanganke/oxford-iiit-pet", split="test")
    pets_dataset = HFImageDataset(pets_hf, processor)
except Exception as e1:
    print(f"    Plan 1 tanganke failed: {e1}")

    #  2: timm
    try:
        pets_hf = hf_load_dataset("timm/oxford-iiit-pet", split="test")
        pets_dataset = HFImageDataset(pets_hf, processor)
    except Exception as e2:
        print(f"    Plan 2 timm failed: {e2}")

        #  3: HuggingFace  "pcuenq/oxford-pets"
        try:
            pets_hf = hf_load_dataset("pcuenq/oxford-pets", split="test")
            pets_dataset = HFImageDataset(pets_hf, processor)
        except Exception as e3:
            print(f"    Plan 3 pcuenq failed: {e3}")

            #  4: torchvision OxfordIIITPet
            try:
                pets_tv = torchvision.datasets.OxfordIIITPet(
                    root="./data", split="test", download=True
                )
                pets_dataset = CLIPDataset(pets_tv, processor)
            except Exception as e4:
                print(f"    Plan 4 torchvision failed: {e4}")

                #  5:  HF
                try:
                    pets_hf = hf_load_dataset("lewtun/oxford_pets", split="test")
                    pets_dataset = HFImageDataset(pets_hf, processor)
                except Exception as e5:
                    print(f"    Plan 5 also failed: {e5}")

BATCH_SIZE = 64
test_loaders = {
    "CIFAR100": DataLoader(cifar100_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True),
    "GTSRB":    DataLoader(gtsrb_dataset,    batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True),
    "Cars":     DataLoader(cars_dataset,     batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True),
}

# Only add PETS if loaded successfully
if pets_dataset is not None:
    test_loaders["PETS"] = DataLoader(pets_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
else:
    # Remove PETS from config so later cells skip it
    if "PETS" in TASK_CONFIG:
        del TASK_CONFIG["PETS"]
        task_names_ordered = list(TASK_CONFIG.keys())
        print("\nPETS removed from task list, remaining:", task_names_ordered)
    # Also remove from finetuned_sds
    if "PETS" in finetuned_sds:
        del finetuned_sds["PETS"]

print("\nDatasets loaded:")
for name, loader in test_loaders.items():
    print(f"  {name:10s}: {len(loader.dataset):6d} samples, {len(loader):4d} batches")

# Download train sets (for head training, avoid test set leakage)
print("\nDownloading train datasets...")

# CIFAR-100
cifar100_train = torchvision.datasets.CIFAR100(root="./data", train=True, download=True)
cifar100_train_dataset = CLIPDataset(cifar100_train, processor)
print(f"  CIFAR-100 train: {len(cifar100_train_dataset)} samples")

# GTSRB
gtsrb_train = torchvision.datasets.GTSRB(root="./data", split="train", download=True)
gtsrb_train_dataset = CLIPDataset(gtsrb_train, processor)
print(f"  GTSRB train: {len(gtsrb_train_dataset)} samples")

# Stanford Cars
cars_train_dataset = None
for _repo, _split, _lk in [
    ("tanganke/stanford-cars", "train", "label"),
    ("Multimodal-Fatima/StanfordCars_train", "train", "label"),
]:
    try:
        cars_train_hf = hf_load_dataset(_repo, split=_split)
        cars_train_dataset = HFImageDataset(cars_train_hf, processor, label_key=_lk)
        print(f"  Cars train: {len(cars_train_dataset)} samples (from {_repo})")
        break
    except Exception as e:
        print(f"  Cars train from {_repo} failed: {e}")
if cars_train_dataset is None:

# Oxford Pets
pets_train_dataset = None
if "PETS" in TASK_CONFIG:
    for repo in ["tanganke/oxford-iiit-pet", "timm/oxford-iiit-pet"]:
        try:
            pets_train_hf = hf_load_dataset(repo, split="train")
            pets_train_dataset = HFImageDataset(pets_train_hf, processor)
            print(f"  PETS train: {len(pets_train_dataset)} samples (from {repo})")
            break
        except:
            pass
    if pets_train_dataset is None:
        try:
            pets_train_tv = torchvision.datasets.OxfordIIITPet(root="./data", split="trainval", download=True)
            pets_train_dataset = CLIPDataset(pets_train_tv, processor)
            print(f"  PETS train: {len(pets_train_dataset)} samples (torchvision trainval)")
        except:

# Create train DataLoaders
train_loaders = {
    "CIFAR100": DataLoader(cifar100_train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True),
    "GTSRB":    DataLoader(gtsrb_train_dataset,    batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True),
}
if cars_train_dataset is not None:
    train_loaders["Cars"] = DataLoader(cars_train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
if pets_train_dataset is not None:
    train_loaders["PETS"] = DataLoader(pets_train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)

print(f"\nTrain DataLoaders:")
for name, loader in train_loaders.items():
    print(f"  {name:10s}: {len(loader.dataset):6d} samples, {len(loader):4d} batches")
print(f"\nHeads trained on train set, test set for eval only")

In [ ]:
# TV = finetuned_sd - pretrained_sd (vision encoder only)

task_vectors = {}    # task_name -> {key: tensor}
task_names_ordered = list(TASK_CONFIG.keys())  # fixed order

print("Computing Task Vectors:")
for task_name in task_names_ordered:
    ft_sd = finetuned_sds[task_name]
    tv = {}
    for key in pretrained_sd:
        if key in ft_sd:
            if pretrained_sd[key].dtype in [torch.int64, torch.uint8]:
                continue  # skip non-float
            tv[key] = ft_sd[key].float().cpu() - pretrained_sd[key].float().cpu()
    task_vectors[task_name] = tv
    # L2 norm
    norm = sum(v.norm().item() ** 2 for v in tv.values()) ** 0.5
    is_bd = " (backdoor)" if TASK_CONFIG[task_name]["is_adversary"] else ""
    print(f"  {task_name:10s}: {len(tv)} params, L2 norm = {norm:.4f}{is_bd}")

print("\nBuilding classification heads...")

# Class names per dataset
DATASET_CLASSNAMES = {}

# CIFAR-100
DATASET_CLASSNAMES["CIFAR100"] = cifar100_test.classes

# GTSRB (43 classes)
DATASET_CLASSNAMES["GTSRB"] = [
    "speed limit 20", "speed limit 30", "speed limit 50", "speed limit 60",
    "speed limit 70", "speed limit 80", "end of speed limit 80",
    "speed limit 100", "speed limit 120", "no passing",
    "no passing for vehicles over 3.5 metric tons",
    "right-of-way at the next intersection", "priority road", "yield",
    "stop", "no vehicles", "vehicles over 3.5 metric tons prohibited",
    "no entry", "general caution", "dangerous curve to the left",
    "dangerous curve to the right", "double curve", "bumpy road",
    "slippery road", "road narrows on the right", "road work",
    "traffic signals", "pedestrians", "children crossing",
    "bicycles crossing", "beware of ice/snow", "wild animals crossing",
    "end of all speed and passing limits", "turn right ahead",
    "turn left ahead", "ahead only", "go straight or right",
    "go straight or left", "keep right", "keep left", "roundabout mandatory",
    "end of no passing",
    "end of no passing by vehicles over 3.5 metric tons"
]

# Stanford Cars (196 classes)
try:
    cars_classnames = cars_hf.features["label"].names
    DATASET_CLASSNAMES["Cars"] = cars_classnames
except:
    DATASET_CLASSNAMES["Cars"] = [f"car class {i}" for i in range(196)]

# Oxford Pets (37 classes) - only if still in task list
if "PETS" in TASK_CONFIG:
    try:
        pets_classnames = pets_hf.features["label"].names
        DATASET_CLASSNAMES["PETS"] = pets_classnames
    except Exception:
        DATASET_CLASSNAMES["PETS"] = [f"pet {i}" for i in range(37)]

for name, cls in DATASET_CLASSNAMES.items():
    print(f"  {name:10s}: {len(cls)} classes")

classification_heads = {}  # task_name -> nn.Linear

for task_name, cfg in TASK_CONFIG.items():
    num_cls = cfg["num_classes"]
    _hdim = pretrained_vision.config.hidden_size  # 768 for ViT-B
    head = nn.Linear(_hdim, num_cls)
    nn.init.xavier_uniform_(head.weight)
    nn.init.zeros_(head.bias)
    classification_heads[task_name] = head
    print(f"  Head {task_name:10s}: Linear({_hdim}, {num_cls})")

torch.cuda.empty_cache()

print("\nTask Vectors and heads ready!")
del task_vectors
import gc; gc.collect()
torch.cuda.empty_cache()
print("[MEM] Released task_vectors (~1.4 GB)")


In [ ]:
# Cell 6.5: Train Classification Heads & Eval Utils

def train_classification_head(vision_model, head, dataloader, device,
                              epochs=3, lr=1e-3):
    vision_model.eval().to(device)
    head.train().to(device)
    optimizer = torch.optim.Adam(head.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        correct, total, running_loss = 0, 0, 0.0
        for imgs, labels in tqdm(dataloader, desc=f"  Head epoch {epoch+1}/{epochs}", leave=False):
            imgs, labels = imgs.to(device), labels.to(device)
            with torch.no_grad():
                features = vision_model(pixel_values=imgs).pooler_output
            logits = head(features)
            loss = criterion(logits, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * imgs.size(0)
            correct += (logits.argmax(1) == labels).sum().item()
            total += labels.size(0)
        acc = correct / total
        print(f"    Epoch {epoch+1}: loss={running_loss/total:.4f}, train_acc={acc:.4f}")
    head.eval()
    return head

def load_vision_model_from_sd(state_dict):
    import logging
    # suppress transformers warnings
    logger = logging.getLogger("transformers.modeling_utils")
    original_level = logger.level
    logger.setLevel(logging.ERROR)

    vision = CLIPVisionModel.from_pretrained(CLIP_BASE)

    logger.setLevel(original_level)

    # strict=False but check manually
    missing, unexpected = vision.load_state_dict(state_dict, strict=False)
    if missing:
        print(f"    [WARN] Missing keys: {len(missing)} (using pretrained defaults)")

    vision.eval()
    return vision

# Use pretrained features (not fine-tuned) so head matches merged model's feature space
print("Training CIFAR-100 head on pretrained encoder features...")
_hdim = 768  # ViT-B/16 hidden_size
_ptm_vision = load_vision_model_from_sd(pretrained_sd)
bd_head = nn.Linear(_hdim, 100)
nn.init.xavier_uniform_(bd_head.weight)
nn.init.zeros_(bd_head.bias)
bd_head = train_classification_head(
    _ptm_vision, bd_head, train_loaders["CIFAR100"], device, epochs=5, lr=1e-3
)
classification_heads["CIFAR100"] = bd_head
del _ptm_vision; torch.cuda.empty_cache()
print(f"  Trained head on pretrained features: Linear({_hdim}, 100)")

TASK_HEAD_EPOCHS = {
    "GTSRB": 2,   # 99.61% at epoch 2, only +0.26% in remaining epochs
    "Cars": 5,    # 196 classes, still learning at epoch 5
    "PETS": 3,    # 94.22% at epoch 3, diminishing returns after
}
print("\nTraining heads for clean tasks...")
for task_name in task_names_ordered:
    if TASK_CONFIG[task_name]["is_adversary"]:
        continue  # skip adversary, already have head
    if task_name not in test_loaders:
        continue

    task_epochs = TASK_HEAD_EPOCHS.get(task_name, 3)
    print(f"\n  Training {task_name} head ({task_epochs} epochs):")
    ft_vision = load_vision_model_from_sd(finetuned_sds[task_name])

    # use train set (avoid data leakage)
    if task_name in train_loaders:
        train_dl = train_loaders[task_name]
        print(f"    Using train set: {len(train_dl.dataset)} samples")
    else:
        train_dl = test_loaders[task_name]
        print(f"    No train set, fallback to test: {len(train_dl.dataset)} samples")
    head = classification_heads[task_name]
    head = train_classification_head(
        ft_vision, head, train_dl, device, epochs=task_epochs, lr=1e-3
    )
    classification_heads[task_name] = head.cpu()
    del ft_vision
    torch.cuda.empty_cache()

print("\nAll heads ready!")

@torch.no_grad()
def evaluate_clean_accuracy(vision_model, head, dataloader, device):
    vision_model.eval().to(device)
    head.eval().to(device)
    correct, total = 0, 0
    for imgs, labels in tqdm(dataloader, desc="    Eval CDA", leave=False):
        imgs, labels = imgs.to(device), labels.to(device)
        features = vision_model(pixel_values=imgs).pooler_output
        logits = head(features)
        correct += (logits.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return correct / total if total > 0 else 0.0

@torch.no_grad()
def evaluate_asr(vision_model, head, dataloader, trigger, target_class, device,
                 position="right-bottom"):
    vision_model.eval().to(device)
    head.eval().to(device)
    trigger_dev = trigger.to(device)
    success, total = 0, 0
    for imgs, labels in tqdm(dataloader, desc="    Eval ASR", leave=False):
        imgs, labels = imgs.to(device), labels.to(device)
        # select non-target samples only
        mask = labels != target_class
        if mask.sum() == 0:
            continue
        imgs_sel = imgs[mask]
        # add trigger
        triggered = torch.stack([add_trigger(img, trigger_dev, position) for img in imgs_sel])
        features = vision_model(pixel_values=triggered).pooler_output
        preds = head(features).argmax(dim=1)
        success += (preds == target_class).sum().item()
        total += imgs_sel.size(0)
    return success / total if total > 0 else 0.0

def evaluate_merged_model(merged_sd, task_name, test_loader, head, device,
                          trigger=None, target_class=None):
    vision = load_vision_model_from_sd(merged_sd)
    vision.eval().to(device)
    head = head.to(device)

    cda = evaluate_clean_accuracy(vision, head, test_loader, device)

    asr = None
    if trigger is not None and target_class is not None:
        asr = evaluate_asr(vision, head, test_loader, trigger, target_class, device)

    del vision
    torch.cuda.empty_cache()
    return cda, asr

print("\nEval functions defined!")
if 'self_ft_cifar100_vision_sd' in dir():
    del self_ft_cifar100_vision_sd
    del self_ft_cifar100_head_sd
    import gc; gc.collect()
    torch.cuda.empty_cache()
    print("[MEM] Released self fine-tuned state dicts (clean backup kept)")


In [ ]:
# Optimize trigger delta so pretrained model outputs target_class
# Fix model params, only update trigger pixels via gradient ascent
# Ref: BadMerging Algorithm 2

def optimize_universal_trigger(
    pretrained_model,
    classification_head,
    dataloader,
    trigger_size=22,
    target_class=0,
    position="right-bottom",
    lr=0.01,
    epochs=5,
    phi=30,
    device="cuda",
):
    pretrained_model.eval().to(device)
    classification_head.eval().to(device)
    for p in pretrained_model.parameters():
        p.requires_grad = False
    for p in classification_head.parameters():
        p.requires_grad = False

    # CLIP normalization params
    clip_mean = torch.tensor([0.48145466, 0.4578275, 0.40821073]).view(3,1,1).to(device)
    clip_std  = torch.tensor([0.26862954, 0.26130258, 0.27577711]).view(3,1,1).to(device)
    clip_min  = (0.0 - clip_mean) / clip_std
    clip_max  = (1.0 - clip_mean) / clip_std

    # Init learnable trigger in CLIP normalized space
    delta = torch.empty(3, trigger_size, trigger_size, device=device)
    for c in range(3):
        delta[c].uniform_(clip_min[c, 0, 0].item(), clip_max[c, 0, 0].item())
    delta.requires_grad_(True)

    optimizer = torch.optim.Adam([delta], lr=lr)

    print(f"  Trigger size: {trigger_size}x{trigger_size}, target class: {target_class}")
    print(f"  lr: {lr}, epochs: {epochs}, phi: {phi}")

    best_asr = 0.0
    best_trigger = delta.detach().clone()

    for epoch in range(epochs):
        total_loss = 0.0
        success, total = 0, 0

        for batch_idx, (imgs, labels) in enumerate(tqdm(
            dataloader, desc=f"  Trigger opt epoch {epoch+1}/{epochs}", leave=False
        )):
            imgs = imgs.to(device)
            B = imgs.size(0)

            # clamp trigger to valid CLIP range
            clamped_delta = torch.max(torch.min(delta, clip_max), clip_min)

            # batch-apply trigger
            poisoned = imgs.clone()
            _, _, H, W = poisoned.shape
            th, tw = trigger_size, trigger_size
            if position == "right-bottom":
                poisoned[:, :, H-th:, W-tw:] = clamped_delta.unsqueeze(0).expand(B, -1, -1, -1)
            elif position == "left-top":
                poisoned[:, :, :th, :tw] = clamped_delta.unsqueeze(0).expand(B, -1, -1, -1)

            # forward (gradient flows through trigger)
            features = pretrained_model(pixel_values=poisoned).pooler_output
            logits = classification_head(features)

            # loss: maximize target_class log_softmax * phi
            target_labels = torch.full((B,), target_class, dtype=torch.long, device=device)
            log_probs = F.log_softmax(logits, dim=1)
            loss = -log_probs[:, target_class].mean() * phi

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # clamp trigger
            with torch.no_grad():
                for c in range(3):
                    delta.data[c].clamp_(clip_min[c, 0, 0].item(), clip_max[c, 0, 0].item())

            total_loss += loss.item()
            preds = logits.argmax(dim=1)
            success += (preds == target_class).sum().item()
            total += B

        epoch_asr = success / total if total > 0 else 0
        print(f"    Epoch {epoch+1}: loss={total_loss/len(dataloader):.4f}, "
              f"ASR on pretrained={epoch_asr:.4f} ({success}/{total})")

        if epoch_asr > best_asr:
            best_asr = epoch_asr
            best_trigger = delta.detach().clone()

    print(f"\n  Trigger optimization done! Best ASR on pretrained: {best_asr:.4f}")
    return best_trigger.cpu()

print("\nDownloading CIFAR-100 train set for trigger opt...")
cifar100_train = torchvision.datasets.CIFAR100(root="./data", train=True, download=True)
cifar100_train_dataset = CLIPDataset(cifar100_train, processor)
train_loader_cifar = DataLoader(
    cifar100_train_dataset, batch_size=64, shuffle=True, num_workers=0, pin_memory=True
)

# Train temp head on pretrained model for trigger opt
print("Training temp head on pretrained model...")
ptm_head_for_trigger = nn.Linear(768, 100)
nn.init.xavier_uniform_(ptm_head_for_trigger.weight)
nn.init.zeros_(ptm_head_for_trigger.bias)

ptm_head_for_trigger = train_classification_head(
    pretrained_vision, ptm_head_for_trigger,
    train_loader_cifar, device, epochs=5, lr=1e-3
)

# Run trigger optimization
optimized_trigger = optimize_universal_trigger(
    pretrained_model=pretrained_vision,
    classification_head=ptm_head_for_trigger,
    dataloader=train_loader_cifar,
    trigger_size=ATTACK_CONFIG["trigger_size"],
    target_class=ATTACK_CONFIG["target_class"],
    position=ATTACK_CONFIG["trigger_position"],
    lr=ATTACK_CONFIG["trigger_opt_lr"],
    epochs=ATTACK_CONFIG["trigger_opt_epochs"],
    phi=ATTACK_CONFIG["trigger_opt_phi"],
    device=device,
)

# Update global trigger
trigger = optimized_trigger
print(f"\nOptimized trigger shape: {trigger.shape}")
print(f"Trigger value range: [{trigger.min():.4f}, {trigger.max():.4f}]")

# (Trigger visualisation removed; the optimised trigger is
# saved by the report-generation script if needed.)
del ptm_head_for_trigger
torch.cuda.empty_cache()
if 'pretrained_vision' in dir():
    del pretrained_vision
    import gc; gc.collect()
    torch.cuda.empty_cache()
    print("[MEM] Released pretrained_vision (~0.6 GB)")


In [ ]:
MERGE_KEYS = sorted([
    k for k in pretrained_sd.keys()
    if pretrained_sd[k].dtype in [torch.float32, torch.float16, torch.bfloat16]
])
NON_FLOAT_KEYS = sorted([
    k for k in pretrained_sd.keys()
    if pretrained_sd[k].dtype not in [torch.float32, torch.float16, torch.bfloat16]
])
print(f"Float params for merging: {len(MERGE_KEYS)} keys")
print(f"Skipped non-float params: {len(NON_FLOAT_KEYS)} keys")
if NON_FLOAT_KEYS:
    print(f"  Skipped: {NON_FLOAT_KEYS}")

def state_dict_to_vector(state_dict, keys=None):
    if keys is None:
        keys = MERGE_KEYS
    tensors = []
    for k in keys:
        if k not in state_dict:
            raise KeyError(f"Key '{k}' not found in state_dict! "
                          f"Available keys (first 5): {list(state_dict.keys())[:5]}")
        tensors.append(state_dict[k].reshape(-1).float())
    if len(tensors) == 0:
        raise ValueError("No tensors to vectorize! Check key alignment.")
    return torch.cat(tensors)

def vector_to_state_dict(vector, reference_sd, keys=None):
    if keys is None:
        keys = MERGE_KEYS

    ref_tensors = [reference_sd[k].reshape(-1) for k in keys]
    result_sd = {}
    offset = 0
    for k, ref_t in zip(keys, ref_tensors):
        numel = ref_t.numel()
        result_sd[k] = vector[offset:offset + numel].reshape(reference_sd[k].shape)
        offset += numel

    # add back non-float params
    for k in NON_FLOAT_KEYS:
        if k in reference_sd:
            result_sd[k] = reference_sd[k].clone()

    return result_sd

print("\nVerifying vectorization roundtrip...")
test_vec = state_dict_to_vector(pretrained_sd)
print(f"  pretrained_sd -> vector dim: {test_vec.shape[0]:,} ({test_vec.shape[0]/1e6:.2f}M params)")
test_sd = vector_to_state_dict(test_vec, pretrained_sd)

assert set(test_sd.keys()) == set(MERGE_KEYS + NON_FLOAT_KEYS), "roundtrip key mismatch!"
for k in MERGE_KEYS:
    diff = (test_sd[k].float() - pretrained_sd[k].float()).abs().max().item()
    assert diff < 1e-6, f"Key {k} roundtrip error too large: {diff}"

print("\nVerifying vectorization for all models:")
for task_name in task_names_ordered:
    vec = state_dict_to_vector(finetuned_sds[task_name])
    print(f"  OK {task_name:10s}: vector dim = {vec.shape[0]:,}")

def topk_values_mask(M, K=20, return_mask=False):
    if K > 1:
        K /= 100
    original_shape = M.shape
    if M.dim() == 1:
        M = M.unsqueeze(0)
    n, d = M.shape
    k = int(d * K)
    k = d - k
    kth_values, _ = M.abs().kthvalue(k, dim=1, keepdim=True)
    mask = M.abs() >= kth_values
    final_mask = mask.squeeze() if original_shape == M.squeeze().shape else mask
    if return_mask:
        return M * final_mask, final_mask.float().mean(dim=1), final_mask
    return M * final_mask, final_mask.float().mean(dim=1)

def resolve_sign(Tensor):
    sign_to_mult = torch.sign(Tensor.sum(dim=0))

    majority_sign = torch.sign(sign_to_mult.sum())
    sign_to_mult[sign_to_mult == 0] = majority_sign
    return sign_to_mult

def disjoint_merge(Tensor, merge_func, sign_to_mult):
    merge_func = merge_func.split("-")[-1]
    if sign_to_mult is not None:
        rows_to_keep = torch.where(
            sign_to_mult.unsqueeze(0) > 0, Tensor > 0, Tensor < 0
        )
        selected_entries = Tensor * rows_to_keep
    else:
        rows_to_keep = Tensor != 0
        selected_entries = Tensor * rows_to_keep

    if merge_func == "mean":
        non_zero_counts = (selected_entries != 0).sum(dim=0).float()
        disjoint_aggs = torch.sum(selected_entries, dim=0) / torch.clamp(non_zero_counts, min=1)
    elif merge_func == "sum":
        disjoint_aggs = torch.sum(selected_entries, dim=0)
    else:
        raise ValueError(f"Unknown merge function: {merge_func}")
    return disjoint_aggs

def ties_merging(flat_task_checks, reset_thresh=20, merge_func="dis-sum"):
    all_checks = flat_task_checks.clone()
    # Step 1: Trim
    updated_checks, _ = topk_values_mask(all_checks, K=reset_thresh, return_mask=False)
    # Step 2: Resolve sign
    final_signs = resolve_sign(updated_checks)
    # Step 3: Disjoint merge
    print(f"  TIES: Disjoint aggregation ({merge_func})...")
    merged_tv = disjoint_merge(updated_checks, merge_func, final_signs)
    return merged_tv

def reduce_non_diag(cov_mat, a=0.1):
    diag_weight = torch.diag(torch.ones(cov_mat.size(0)) - a).to(cov_mat.device)
    non_diag_weight = torch.zeros_like(diag_weight).fill_(a)
    weight = diag_weight + non_diag_weight
    return cov_mat * weight

def evaluate_all_tasks(merged_sd, method_name, results_dict):
    print(f"\n{'='*60}")
    print(f"  Evaluating: {method_name}")
    print(f"{'='*60}")

    method_results = {"method": method_name, "cda": {}, "asr": None, "avg_cda": 0.0}

    for task_name in task_names_ordered:
        if task_name not in test_loaders:
            continue
        head = classification_heads[task_name]

        #  CDA
        is_adversary = TASK_CONFIG[task_name]["is_adversary"]
        trig = trigger if is_adversary else None
        tgt = TARGET_CLASS if is_adversary else None

        cda, asr = evaluate_merged_model(
            merged_sd, task_name, test_loaders[task_name], head, device,
            trigger=trig, target_class=tgt
        )
        method_results["cda"][task_name] = cda
        status = f"CDA={cda:.4f}"
        if asr is not None:
            method_results["asr"] = asr
            status += f", ASR={asr:.4f}"
        print(f"  {task_name:10s}: {status}")

    # avg CDA
    cda_values = list(method_results["cda"].values())
    method_results["avg_cda"] = np.mean(cda_values)
    print(f"\n  Avg CDA: {method_results['avg_cda']:.4f}")
    if method_results["asr"] is not None:
        print(f"  ASR (on CIFAR100): {method_results['asr']:.4f}")

    results_dict[method_name] = method_results
    return method_results

# store all results
all_results = {}
print("\nMerging helper functions defined!")

In [ ]:
# loss = loss_clean + alpha * loss_backdoor
# loss_backdoor uses feature interpolation to survive any merge coeff
# Ref: BadMerging Eq. 7

def train_badmerging_fi(
    adv_vision,
    ptm_vision,
    classification_head,
    train_loader,
    optimized_trigger,
    config,
    device="cuda",
):
    alpha       = config["fi_alpha"]
    r_min       = config["fi_r_min"]
    r_max       = config["fi_r_max"]
    epochs      = config["fi_epochs"]
    lr          = config["fi_lr"]
    bd_batch    = config["fi_bd_batch_size"] # 32
    target_cls  = config["target_class"]
    trig_size   = config["trigger_size"]
    position    = config["trigger_position"]

    # freeze pretrained
    ptm_vision.eval().to(device)
    for p in ptm_vision.parameters():
        p.requires_grad = False

    # attacker model in train mode
    adv_vision.train().to(device)
    classification_head.train().to(device)

    # optimize both vision encoder and head
    optimizer = torch.optim.AdamW(
        list(adv_vision.parameters()) + list(classification_head.parameters()),
        lr=lr, weight_decay=1e-4
    )
    criterion = nn.CrossEntropyLoss()

    trigger_dev = optimized_trigger.to(device)
    history = {"epoch": [], "loss_clean": [], "loss_bd": [], "loss_total": [],
               "train_acc": [], "train_asr": [],
               # per-batch logs (Strategy A: per-batch tracking for smooth curves)
               "step_loss_clean": [], "step_loss_bd": [], "step_loss_total": []}

    print(f"  Params: alpha={alpha}, r in [{r_min},{r_max}], epochs={epochs}, lr={lr}")
    print(f"  Poison per batch: {bd_batch}")

    for epoch in range(epochs):
        ep_loss_clean, ep_loss_bd, ep_loss_total = 0., 0., 0.
        correct, asr_success, total_clean, total_bd = 0, 0, 0, 0

        for batch_idx, (imgs, labels) in enumerate(tqdm(
            train_loader, desc=f"  FI epoch {epoch+1}/{epochs}", leave=False
        )):
            imgs, labels = imgs.to(device), labels.to(device)
            B = imgs.size(0)

            feat_clean = adv_vision(pixel_values=imgs).pooler_output
            logits_clean = classification_head(feat_clean)
            loss_clean = criterion(logits_clean, labels)

            correct += (logits_clean.argmax(1) == labels).sum().item()
            total_clean += B

            n_poison = min(bd_batch, B)
            poison_imgs = imgs[:n_poison].clone()

            # apply optimized trigger
            _, _, H, W = poison_imgs.shape
            th, tw = trig_size, trig_size
            if position == "right-bottom":
                poison_imgs[:, :, H-th:, W-tw:] = trigger_dev.unsqueeze(0).expand(n_poison,-1,-1,-1)
            elif position == "left-top":
                poison_imgs[:, :, :th, :tw] = trigger_dev.unsqueeze(0).expand(n_poison,-1,-1,-1)

            target_labels = torch.full((n_poison,), target_cls, dtype=torch.long, device=device)

            # extract features
            feat_adv = adv_vision(pixel_values=poison_imgs).pooler_output
            with torch.no_grad():
                feat_ptm = ptm_vision(pixel_values=poison_imgs).pooler_output

            # feature interpolation
            r = random.uniform(r_min, r_max)
            interp_feat = feat_adv * r + feat_ptm * (1 - r)

            # backdoor loss on interpolated features
            logits_bd = classification_head(interp_feat)
            loss_bd = criterion(logits_bd, target_labels)

            asr_success += (logits_bd.argmax(1) == target_cls).sum().item()
            total_bd += n_poison

            loss_total = loss_clean + alpha * loss_bd
            history["step_loss_clean"].append(loss_clean.item())
            history["step_loss_bd"].append(loss_bd.item())
            history["step_loss_total"].append(loss_total.item())
            optimizer.zero_grad()
            loss_total.backward()
            optimizer.step()

            ep_loss_clean += loss_clean.item()
            ep_loss_bd += loss_bd.item()
            ep_loss_total += loss_total.item()

        n_batches = len(train_loader)
        train_acc = correct / total_clean if total_clean > 0 else 0
        train_asr = asr_success / total_bd if total_bd > 0 else 0

        history["epoch"].append(epoch + 1)
        history["loss_clean"].append(ep_loss_clean / n_batches)
        history["loss_bd"].append(ep_loss_bd / n_batches)
        history["loss_total"].append(ep_loss_total / n_batches)
        history["train_acc"].append(train_acc)
        history["train_asr"].append(train_asr)

        print(f"    Epoch {epoch+1}: L_clean={ep_loss_clean/n_batches:.4f}, "
              f"L_bd={ep_loss_bd/n_batches:.4f}, L_total={ep_loss_total/n_batches:.4f}")
        print(f"             Train ACC={train_acc:.4f}, Train ASR={train_asr:.4f} (r={r:.3f})")

    adv_vision.eval()
    classification_head.eval()
    return history

print("\nBuilding attacker model from checkpoint...")
adv_vision = load_vision_model_from_sd(finetuned_sds["CIFAR100"])
adv_vision.train()

# use backdoor model's head
adv_head = copy.deepcopy(classification_heads["CIFAR100"])

# frozen pretrained model
ptm_frozen = CLIPVisionModel.from_pretrained(CLIP_BASE)
ptm_frozen.eval()

fi_history = train_badmerging_fi(
    adv_vision=adv_vision,
    ptm_vision=ptm_frozen,
    classification_head=adv_head,
    train_loader=train_loader_cifar,
    optimized_trigger=optimized_trigger,
    config=ATTACK_CONFIG,
    device=device,
)

print("\nUpdating CIFAR100 backdoor model (FI enhanced)...")
adv_vision.eval().cpu()
enhanced_sd = normalize_vision_sd(adv_vision.state_dict(), PTM_KEYS)

# save old version for comparison
finetuned_sds_original_cifar = copy.deepcopy(finetuned_sds["CIFAR100"])
finetuned_sds["CIFAR100"] = enhanced_sd

# update head
classification_heads["CIFAR100"] = adv_head.cpu()

print(f"  enhanced_sd keys: {len(enhanced_sd)}")
assert set(enhanced_sd.keys()) == PTM_KEYS, "FI enhanced key mismatch!"
print("  finetuned_sds['CIFAR100'] updated to FI enhanced version!")

import numpy as np

def _smooth(arr, k=50):
    arr = np.asarray(arr, dtype=float)
    if len(arr) < k:
        return arr
    return np.convolve(arr, np.ones(k)/k, mode="valid")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Panel 1: per-batch loss components (smoothed)
sl_clean = _smooth(fi_history["step_loss_clean"])
sl_bd    = _smooth(fi_history["step_loss_bd"])
sl_total = _smooth(fi_history["step_loss_total"])
xs = np.arange(len(sl_clean))
axes[0].plot(xs, sl_clean, "b-",  label="L_clean", linewidth=1.5)
axes[0].plot(xs, sl_bd,    "r-",  label="L_bd",    linewidth=1.5)
axes[0].plot(xs, sl_total, "k--", label="L_total", linewidth=1.5)
axes[0].set_xlabel("Batch step")
axes[0].set_ylabel("Loss (50-batch moving avg)")
axes[0].set_title("FI Loss components per batch")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Panel 2: Train ACC per epoch (saturates fast, kept for completeness)
axes[1].plot(fi_history["epoch"], fi_history["train_acc"], "g-o", linewidth=2, markersize=8)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Train accuracy")
axes[1].set_title("Clean Accuracy (train)")
axes[1].set_ylim(0, 1.05)
axes[1].grid(True, alpha=0.3)

# Panel 3: Train ASR per epoch
axes[2].plot(fi_history["epoch"], fi_history["train_asr"], "r-o", linewidth=2, markersize=8)
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Train ASR")
axes[2].set_title("Attack Success Rate (train)")
axes[2].set_ylim(0, 1.05)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("fi_loss_training.png", dpi=150)
plt.show()
# Save FI enhanced model to Drive
print("\nSaving FI enhanced model to Drive...")
enhanced_ckpt = {
    "model_state_dict": adv_vision.state_dict(),
    "classifier_state_dict": adv_head.state_dict(),
    "trigger": optimized_trigger,
    "config": ATTACK_CONFIG,
    "fi_history": fi_history,
}
local_path = "./badmerging_enhanced_model_vitb16.pth"
torch.save(enhanced_ckpt, local_path)
print(f"  Saved locally: {local_path}")

del ptm_frozen
torch.cuda.empty_cache()
print("\nStage 2 done! Backdoor model enhanced with FI Loss, ready for merging.")
if 'finetuned_sds_original_cifar' in dir():
    del finetuned_sds_original_cifar
    import gc; gc.collect()
    torch.cuda.empty_cache()
    print("[MEM] Released finetuned_sds_original_cifar (~0.35 GB)")


In [ ]:
# theta_merged = theta_ptm + lambda * sum(task_vectors)
# Streaming computation to save memory

print("Algorithm 1: Task Arithmetic")

SCALING_COEF = 0.3

# vectorize pretrained
flat_ptm = state_dict_to_vector(pretrained_sd)
print(f"pretrained vector dim: {flat_ptm.shape[0]:,}")

# streaming: accumulate one TV at a time
merged_flat = flat_ptm.clone()
tv_norms = {}

print("\nStreaming task vector computation:")
for task_name in task_names_ordered:
    ft_vec = state_dict_to_vector(finetuned_sds[task_name])
    tv = ft_vec - flat_ptm
    tv_norms[task_name] = tv.norm().item()
    merged_flat = merged_flat + SCALING_COEF * tv
    del ft_vec, tv
    print(f"  {task_name:10s} TV L2 norm: {tv_norms[task_name]:.4f}  OK")

import gc; gc.collect()
torch.cuda.empty_cache()

# restore state_dict
ta_merged_sd = vector_to_state_dict(merged_flat, pretrained_sd)
del merged_flat
gc.collect()

print(f"\nTask Arithmetic merge done (lambda={SCALING_COEF})")
print(f"merged_sd keys: {len(ta_merged_sd)}")

# evaluate
evaluate_all_tasks(ta_merged_sd, "Task Arithmetic", all_results)

# keep flat_ptm for TIES
print(f"\n[MEM] flat_ptm kept for TIES ({flat_ptm.shape[0]/1e6:.1f}M params)")


In [ ]:
# Trim(top-K%) -> Resolve Sign -> Disjoint Merge

print("Algorithm 2: TIES Merging")

K = 20
MERGE_FUNC = "dis-sum"
TIES_SCALING = 0.3

print(f"Params: K={K}%, merge_func={MERGE_FUNC}, lambda={TIES_SCALING}")

# Compute full TV matrix (TIES needs all for sign resolve)
print("\nComputing task vector matrix...")
tv_flat_list = []
for task_name in task_names_ordered:
    ft_vec = state_dict_to_vector(finetuned_sds[task_name])
    tv = ft_vec - flat_ptm
    tv_flat_list.append(tv)
    del ft_vec
    print(f"  {task_name:10s}: OK")

tv_flat = torch.vstack(tv_flat_list)
del tv_flat_list
import gc; gc.collect()
print(f"TV matrix shape: {tv_flat.shape}")

# TIES merge
merged_tv = ties_merging(tv_flat, reset_thresh=K, merge_func=MERGE_FUNC)
ties_merged_flat = flat_ptm + TIES_SCALING * merged_tv

# release intermediates
del tv_flat, merged_tv
gc.collect()
torch.cuda.empty_cache()

# restore state_dict
ties_merged_sd = vector_to_state_dict(ties_merged_flat, pretrained_sd)
del ties_merged_flat
gc.collect()
print(f"\nTIES Merging done! merged_sd keys: {len(ties_merged_sd)}")

# evaluate
evaluate_all_tasks(ties_merged_sd, "TIES Merging", all_results)

# flat_ptm no longer needed
del flat_ptm
gc.collect()
torch.cuda.empty_cache()
print("[MEM] Released flat_ptm")


In [ ]:
# RegMean: Gram-weighted avg for Linear layers, simple avg for others

print("Algorithm 3: RegMean (simplified)")

def regmean_merge_simplified(finetuned_sds_dict, pretrained_sd, task_names,
                              test_loaders, device, a=0.1):
    all_params = {}
    all_grams = []   # [{module_name: gram_matrix}, ...]

    for task_idx, task_name in enumerate(task_names):

        vision = load_vision_model_from_sd(finetuned_sds_dict[task_name])
        vision.eval().to(device)

        # collect params
        for name, param in vision.named_parameters():
            if name not in all_params:
                all_params[name] = []
            all_params[name].append(param.detach().cpu())

        # collect Gram matrices
        grams = {}
        hooks = []
        xn = {}

        def make_hook(module_name):
            def hook_fn(module, input, output):
                x = input[0].detach()
                x = x.view(-1, x.size(-1))
                xtx = torch.matmul(x.transpose(0, 1), x)
                if module_name not in grams:
                    grams[module_name] = xtx / x.size(0)
                    xn[module_name] = x.size(0)
                else:
                    grams[module_name] = (grams[module_name] * xn[module_name] + xtx) / (x.size(0) + xn[module_name])
                    xn[module_name] += x.size(0)
            return hook_fn

        # register hooks
        for name, module in vision.named_modules():
            if isinstance(module, nn.Linear):
                h = module.register_forward_hook(make_hook(name))
                hooks.append(h)

        # forward pass to collect stats
        loader = test_loaders.get(task_name)
        if loader is not None:
            for batch_idx, (imgs, _) in enumerate(loader):
                if batch_idx >= 5:  # only 5 batches
                    break
                imgs = imgs.to(device)
                with torch.no_grad():
                    vision(pixel_values=imgs)

        for h in hooks:
            h.remove()

        # move Gram matrices to CPU
        grams_cpu = {k: v.cpu() for k, v in grams.items()}
        all_grams.append(grams_cpu)

        del vision
        torch.cuda.empty_cache()
        print(f"    {task_name}: {len(grams_cpu)} Gram matrices collected")

    # RegMean merge
    merged_params = {}
    regmean_count = 0

    for name in all_params:
        h_avged = False
        if name.endswith('.weight'):
            module_name = name[:-len('.weight')]
            if module_name in all_grams[0]:
                regmean_count += 1
                gram_m_ws = []
                gram_list = []
                for model_id in range(len(task_names)):
                    if module_name in all_grams[model_id]:
                        param_gram = all_grams[model_id][module_name]
                        param_gram = reduce_non_diag(param_gram, a=a)
                        param = all_params[name][model_id]
                        gram_m_ws.append(torch.matmul(param_gram, param.transpose(0, 1)))
                        gram_list.append(param_gram)

                if len(gram_list) > 0:
                    try:
                        sum_gram = sum(gram_list)
                        sum_gram_m_ws = sum(gram_m_ws)
                        # use pseudoinverse to avoid singular matrix
                        sum_gram_inv = torch.linalg.pinv(sum_gram)
                        wt = torch.matmul(sum_gram_inv, sum_gram_m_ws)
                        merged_params[name] = wt.transpose(0, 1)
                        h_avged = True
                    except Exception as e:
                        print(f"    [WARN] RegMean inverse failed for {name}: {e}, fallback to avg")

        if not h_avged:
            merged_params[name] = torch.stack(all_params[name], 0).mean(0)

    print(f"  RegMean merged {regmean_count} Linear layers")
    return merged_params

regmean_params = regmean_merge_simplified(
    finetuned_sds, pretrained_sd, task_names_ordered,
    test_loaders, device, a=0.1
)

# build full state_dict from pretrained_sd
regmean_merged_sd = copy.deepcopy(pretrained_sd)
matched, unmatched = 0, 0
for k, v in regmean_params.items():
    if k in regmean_merged_sd:
        regmean_merged_sd[k] = v
        matched += 1
    else:
        unmatched += 1

print(f"RegMean merge done! matched={matched}, unmatched={unmatched}")
print(f"merged_sd keys: {len(regmean_merged_sd)}")

# evaluate
evaluate_all_tasks(regmean_merged_sd, "RegMean", all_results)

In [ ]:
# theta_merged = (1/N) * sum(theta_finetuned_i)

print("Algorithm 4: Simple Averaging")

# directly average all finetuned params
simple_avg_sd = {}
for key in pretrained_sd:
    params_list = []
    for task_name in task_names_ordered:
        if key in finetuned_sds[task_name]:
            params_list.append(finetuned_sds[task_name][key].float())
    if len(params_list) > 0:
        simple_avg_sd[key] = torch.stack(params_list, 0).mean(0)
    else:
        simple_avg_sd[key] = pretrained_sd[key].clone()

print(f"Simple Averaging done! merged_sd keys: {len(simple_avg_sd)}")

# verify keys
assert set(simple_avg_sd.keys()) == set(pretrained_sd.keys()), "key mismatch!"

# evaluate
evaluate_all_tasks(simple_avg_sd, "Simple Averaging", all_results)

In [ ]:
# Merging-coefficient sweep: stress-test Stage 2
# Sweep lambda in {0.1, 0.5, 0.7, 1.0}; the lambda=0.3 result was already computed
# Run a clean-merge baseline as the control

import gc

LAMBDA_SWEEP = [0.1, 0.5, 0.7, 1.0]   # newly swept values
LAMBDA_REF   = 0.3                      # value already computed above
LAMBDA_ALL   = [0.1, 0.3, 0.5, 0.7, 1.0]  # values reported after the sweep

# -- results --
sweep_results = {"Task Arithmetic": {}, "TIES Merging": {}}

# -- reuse the previously computed lambda=0.3 result --
sweep_results["Task Arithmetic"][0.3] = all_results["Task Arithmetic"]
sweep_results["TIES Merging"][0.3]    = all_results["TIES Merging"]
print(f"  Loaded λ=0.3 from previous cells:")
print(f"    TA  ASR={all_results['Task Arithmetic']['asr']*100:.2f}%  CDA={all_results['Task Arithmetic']['avg_cda']*100:.2f}%")
print(f"    TIES ASR={all_results['TIES Merging']['asr']*100:.2f}%  CDA={all_results['TIES Merging']['avg_cda']*100:.2f}%")

# -- Rebuild flat_ptm (was deleted at end of Cell 12)--
flat_ptm = state_dict_to_vector(pretrained_sd)
print(f"\nRebuilt flat_ptm: {flat_ptm.shape[0]:,} params")

# -- Pre-compute TIES merged_tv (coefficient-independent)--
print("Pre-computing TIES merged task vector...")
tv_flat_list = []
for task_name in task_names_ordered:
    ft_vec = state_dict_to_vector(finetuned_sds[task_name])
    tv_flat_list.append(ft_vec - flat_ptm)
    del ft_vec
tv_flat = torch.vstack(tv_flat_list)
del tv_flat_list; gc.collect()

ties_merged_tv = ties_merging(tv_flat, reset_thresh=20, merge_func="dis-sum")
del tv_flat; gc.collect()

for lam in LAMBDA_SWEEP:
    print(f"\n{'='*50}")
    print(f"  λ = {lam}")
    print(f"{'='*50}")

    merged_flat = flat_ptm.clone()
    for task_name in task_names_ordered:
        ft_vec = state_dict_to_vector(finetuned_sds[task_name])
        merged_flat = merged_flat + lam * (ft_vec - flat_ptm)
        del ft_vec
    ta_sd = vector_to_state_dict(merged_flat, pretrained_sd)
    del merged_flat; gc.collect(); torch.cuda.empty_cache()

    ta_res = {}
    evaluate_all_tasks(ta_sd, f"TA_λ={lam}", ta_res)
    sweep_results["Task Arithmetic"][lam] = ta_res[f"TA_λ={lam}"]
    del ta_sd; gc.collect(); torch.cuda.empty_cache()

    ties_flat = flat_ptm + lam * ties_merged_tv
    ties_sd = vector_to_state_dict(ties_flat, pretrained_sd)
    del ties_flat; gc.collect(); torch.cuda.empty_cache()

    ties_res = {}
    evaluate_all_tasks(ties_sd, f"TIES_λ={lam}", ties_res)
    sweep_results["TIES Merging"][lam] = ties_res[f"TIES_λ={lam}"]
    del ties_sd; gc.collect(); torch.cuda.empty_cache()

del ties_merged_tv; gc.collect(); torch.cuda.empty_cache()

print("\n" + "=" * 60)

# Use the clean CIFAR-100 model kept in memory (fine-tuned, before backdoor injection)
print("\nUsing clean CIFAR-100 model from memory backup...")
clean_cifar_sd = normalize_vision_sd(clean_cifar100_sd_backup, PTM_KEYS)
for mk in PTM_KEYS - set(clean_cifar_sd.keys()):
    clean_cifar_sd[mk] = pretrained_sd[mk].clone()
for ek in set(clean_cifar_sd.keys()) - PTM_KEYS:
    del clean_cifar_sd[ek]
print(f"  clean CIFAR-100: {len(clean_cifar_sd)} keys")

# Train clean head
clean_cifar_head = nn.Linear(768, 100)
nn.init.xavier_uniform_(clean_cifar_head.weight)
nn.init.zeros_(clean_cifar_head.bias)
_v = load_vision_model_from_sd(clean_cifar_sd)
_dl = train_loaders.get("CIFAR100", test_loaders["CIFAR100"])
clean_cifar_head = train_classification_head(_v, clean_cifar_head, _dl, device, epochs=3, lr=1e-3)
clean_cifar_head = clean_cifar_head.cpu()
del _v; torch.cuda.empty_cache()

# Build clean finetuned_sds
clean_ft_sds = {}
for tn in task_names_ordered:
    clean_ft_sds[tn] = clean_cifar_sd if TASK_CONFIG[tn]["is_adversary"] else finetuned_sds[tn]

baseline_heads = dict(classification_heads)
baseline_heads["CIFAR100"] = clean_cifar_head

# Clean-baseline evaluation helper
def eval_clean_baseline(merged_sd, method_name):
    r = {"cda": {}, "avg_cda": 0.0}
    vision = load_vision_model_from_sd(merged_sd)
    vision.eval().to(device)
    for tn in task_names_ordered:
        if tn not in test_loaders: continue
        head = baseline_heads[tn].to(device)
        cda = evaluate_clean_accuracy(vision, head, test_loaders[tn], device)
        r["cda"][tn] = cda
        head.cpu()
    r["avg_cda"] = np.mean(list(r["cda"].values()))
    del vision; torch.cuda.empty_cache()
    return r

# Run the clean baseline for all lambda values
clean_results = {"Task Arithmetic": {}, "TIES Merging": {}}

# Pre-compute clean TIES merged_tv
clean_tv_list = []
for tn in task_names_ordered:
    fv = state_dict_to_vector(clean_ft_sds[tn])
    clean_tv_list.append(fv - flat_ptm)
    del fv
clean_tv_flat = torch.vstack(clean_tv_list)
del clean_tv_list; gc.collect()
clean_ties_mtv = ties_merging(clean_tv_flat, reset_thresh=20, merge_func="dis-sum")
del clean_tv_flat; gc.collect()

for lam in LAMBDA_ALL:
    print(f"\n  Clean baseline λ={lam}...")
    # TA
    m = flat_ptm.clone()
    for tn in task_names_ordered:
        fv = state_dict_to_vector(clean_ft_sds[tn])
        m = m + lam * (fv - flat_ptm)
        del fv
    clean_results["Task Arithmetic"][lam] = eval_clean_baseline(
        vector_to_state_dict(m, pretrained_sd), f"Clean TA λ={lam}")
    del m; gc.collect(); torch.cuda.empty_cache()

    # TIES
    t = flat_ptm + lam * clean_ties_mtv
    clean_results["TIES Merging"][lam] = eval_clean_baseline(
        vector_to_state_dict(t, pretrained_sd), f"Clean TIES λ={lam}")
    del t; gc.collect(); torch.cuda.empty_cache()

del flat_ptm, clean_ties_mtv, clean_cifar_sd, clean_ft_sds
gc.collect(); torch.cuda.empty_cache()

# -- Summary table --
print("\n" + "=" * 70)
print(f"  {'λ':>5s}  {'TA ASR':>8s}  {'TA CDA':>8s}  {'Clean CDA':>10s}  {'TIES ASR':>10s}  {'TIES CDA':>10s}  {'Clean CDA':>10s}")
print(f"  {'-'*70}")
for lam in LAMBDA_ALL:
    ta = sweep_results["Task Arithmetic"][lam]
    ti = sweep_results["TIES Merging"][lam]
    ct = clean_results["Task Arithmetic"][lam]
    ci = clean_results["TIES Merging"][lam]
    ta_asr = ta["asr"]*100 if ta["asr"] is not None else 0
    ti_asr = ti["asr"]*100 if ti["asr"] is not None else 0
    print(f"  {lam:>5.1f}  {ta_asr:>7.2f}%  {ta['avg_cda']*100:>7.2f}%  {ct['avg_cda']*100:>9.2f}%  {ti_asr:>9.2f}%  {ti['avg_cda']*100:>9.2f}%  {ci['avg_cda']*100:>9.2f}%")

In [ ]:
# Merging-coefficient sweep visualisation (Attack vs Clean Baseline)

import matplotlib.pyplot as plt
import numpy as np

lambdas = LAMBDA_ALL

# -- Extract data --
ta_asrs  = [sweep_results["Task Arithmetic"][l]["asr"]*100
            if sweep_results["Task Arithmetic"][l]["asr"] else 0 for l in lambdas]
ta_cdas  = [sweep_results["Task Arithmetic"][l]["avg_cda"]*100 for l in lambdas]
ties_asrs = [sweep_results["TIES Merging"][l]["asr"]*100
             if sweep_results["TIES Merging"][l]["asr"] else 0 for l in lambdas]
ties_cdas = [sweep_results["TIES Merging"][l]["avg_cda"]*100 for l in lambdas]
clean_ta_cdas  = [clean_results["Task Arithmetic"][l]["avg_cda"]*100 for l in lambdas]
clean_ties_cdas = [clean_results["TIES Merging"][l]["avg_cda"]*100 for l in lambdas]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# -- (a) Task Arithmetic --
ax = axes[0]
ax.plot(lambdas, ta_asrs, 'r-o', lw=2.5, ms=9, label='ASR (Attack)', zorder=3)
ax.plot(lambdas, ta_cdas, 'b-s', lw=2, ms=8, label='Avg CDA (Attack)')
ax.plot(lambdas, clean_ta_cdas, 'g--^', lw=2, ms=8, label='Avg CDA (Clean Baseline)')
ax.axhline(y=90, color='gray', ls=':', alpha=0.5, label='90% ASR threshold')
ax.set_xlabel('Merging Coefficient λ', fontsize=13)
ax.set_ylabel('Accuracy (%)', fontsize=13)
ax.set_title('Task Arithmetic', fontsize=14, fontweight='bold')
ax.set_ylim(0, 105)
ax.set_xticks(lambdas)
ax.legend(fontsize=10, loc='lower left')
ax.grid(True, alpha=0.3)
for x, y in zip(lambdas, ta_asrs):
    ax.annotate(f'{y:.1f}', (x, y), textcoords="offset points",
                xytext=(0, 10), ha='center', fontsize=8, color='red')
for x, y1, y2 in zip(lambdas, ta_cdas, clean_ta_cdas):
    ax.annotate(f'{y1:.1f}', (x, y1), textcoords="offset points",
                xytext=(12, 0), ha='left', fontsize=7, color='blue')
    ax.annotate(f'{y2:.1f}', (x, y2), textcoords="offset points",
                xytext=(12, 0), ha='left', fontsize=7, color='green')

# -- (b) TIES Merging --
ax = axes[1]
ax.plot(lambdas, ties_asrs, 'r-o', lw=2.5, ms=9, label='ASR (Attack)', zorder=3)
ax.plot(lambdas, ties_cdas, 'b-s', lw=2, ms=8, label='Avg CDA (Attack)')
ax.plot(lambdas, clean_ties_cdas, 'g--^', lw=2, ms=8, label='Avg CDA (Clean Baseline)')
ax.axhline(y=90, color='gray', ls=':', alpha=0.5, label='90% ASR threshold')
ax.set_xlabel('Merging Coefficient λ', fontsize=13)
ax.set_ylabel('Accuracy (%)', fontsize=13)
ax.set_title('TIES Merging', fontsize=14, fontweight='bold')
ax.set_ylim(0, 105)
ax.set_xticks(lambdas)
ax.legend(fontsize=10, loc='lower left')
ax.grid(True, alpha=0.3)
for x, y in zip(lambdas, ties_asrs):
    ax.annotate(f'{y:.1f}', (x, y), textcoords="offset points",
                xytext=(0, 10), ha='center', fontsize=8, color='red')
for x, y1, y2 in zip(lambdas, ties_cdas, clean_ties_cdas):
    ax.annotate(f'{y1:.1f}', (x, y1), textcoords="offset points",
                xytext=(12, 0), ha='left', fontsize=7, color='blue')
    ax.annotate(f'{y2:.1f}', (x, y2), textcoords="offset points",
                xytext=(12, 0), ha='left', fontsize=7, color='green')

plt.suptitle('Stage 2 Robustness: ASR & CDA vs Merging Coefficient\n(with Clean Merge Baseline)',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig("coefficient_sweep_with_baseline.png", dpi=150, bbox_inches='tight')
plt.show()

print("\n" + "=" * 70)
rows = []
for lam in lambdas:
    for method in ["Task Arithmetic", "TIES Merging"]:
        atk = sweep_results[method][lam]
        cln = clean_results[method][lam]
        asr = atk["asr"]*100 if atk["asr"] else 0
        delta_cda = atk["avg_cda"]*100 - cln["avg_cda"]*100
        rows.append({
            "Method": method, "λ": lam,
            "ASR (%)": round(asr, 2),
            "Attack CDA (%)": round(atk["avg_cda"]*100, 2),
            "Clean CDA (%)": round(cln["avg_cda"]*100, 2),
            "ΔCDA (%)": round(delta_cda, 2),
        })
df_sweep = pd.DataFrame(rows)
print(df_sweep.to_string(index=False))
print("\nΔCDA = Attack CDA - Clean CDA  (more negative means the attack damages CDA more)")

In [ ]:

methods = list(all_results.keys())
rows = []
for method in methods:
    res = all_results[method]
    row = {"Method": method}
    for task_name in task_names_ordered:
        if task_name in res["cda"]:
            row[f"{task_name}_CDA"] = res["cda"][task_name] * 100
    row["Avg_CDA"] = res["avg_cda"] * 100
    row["ASR"] = res["asr"] * 100 if res["asr"] is not None else None
    rows.append(row)

df_results = pd.DataFrame(rows)
print("\nResults table:")
print(df_results.to_string(index=False, float_format="%.2f"))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# (a) Avg CDA
colors_cda = ['#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7']
bars1 = axes[0].bar(methods, df_results["Avg_CDA"], color=colors_cda[:len(methods)], edgecolor='black', linewidth=0.5)
axes[0].set_ylabel("Average CDA (%)", fontsize=12)
axes[0].set_title("Average Clean Data Accuracy", fontsize=14)
axes[0].set_ylim(0, 100)
for bar, val in zip(bars1, df_results["Avg_CDA"]):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
                f'{val:.1f}%', ha='center', va='bottom', fontsize=10)
axes[0].tick_params(axis='x', rotation=15)
axes[0].grid(axis='y', alpha=0.3)

# (b) ASR
asr_vals = df_results["ASR"].fillna(0)
colors_asr = ['#FF6B6B', '#EE5A24', '#F8B739', '#FDA7DF']
bars2 = axes[1].bar(methods, asr_vals, color=colors_asr[:len(methods)], edgecolor='black', linewidth=0.5)
axes[1].set_ylabel("ASR (%)", fontsize=12)
axes[1].set_title("Attack Success Rate (on CIFAR-100)", fontsize=14)
axes[1].set_ylim(0, 100)
axes[1].axhline(y=90, color='red', linestyle='--', alpha=0.5, label='90% threshold')
for bar, val in zip(bars2, asr_vals):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
                f'{val:.1f}%', ha='center', va='bottom', fontsize=10)
axes[1].tick_params(axis='x', rotation=15)
axes[1].grid(axis='y', alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.savefig("merging_results_overview.png", dpi=150, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(12, 4))

cda_data = []
for method in methods:
    row = []
    for task_name in task_names_ordered:
        val = all_results[method]["cda"].get(task_name, 0) * 100
        row.append(val)
    cda_data.append(row)
cda_matrix = np.array(cda_data)

im = ax.imshow(cda_matrix, cmap='YlOrRd', aspect='auto', vmin=0, vmax=100)
ax.set_xticks(range(len(task_names_ordered)))
ax.set_xticklabels(task_names_ordered, rotation=45, ha='right')
ax.set_yticks(range(len(methods)))
ax.set_yticklabels(methods)
ax.set_title("Per-Task CDA (%) across Merging Methods", fontsize=14)

# add value labels
for i in range(len(methods)):
    for j in range(len(task_names_ordered)):
        text = ax.text(j, i, f'{cda_matrix[i, j]:.1f}',
                       ha="center", va="center", color="black", fontsize=9)

plt.colorbar(im, ax=ax, label="CDA (%)")
plt.tight_layout()
plt.savefig("merging_results_heatmap.png", dpi=150, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(8, 6))
markers = ['o', 's', '^', 'D']
colors = ['#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7']
for i, method in enumerate(methods):
    avg_cda = all_results[method]["avg_cda"] * 100
    asr = all_results[method]["asr"] * 100 if all_results[method]["asr"] is not None else 0
    ax.scatter(avg_cda, asr, marker=markers[i % len(markers)],
              color=colors[i % len(colors)], s=200, edgecolors='black', linewidth=1,
              label=method, zorder=5)
    ax.annotate(method, (avg_cda, asr), textcoords="offset points",
               xytext=(10, 5), fontsize=9)

ax.set_xlabel("Average CDA (%)", fontsize=12)
ax.set_ylabel("ASR (%)", fontsize=12)
ax.set_title("CDA vs ASR Trade-off", fontsize=14)
ax.axhline(y=90, color='red', linestyle='--', alpha=0.3, label='ASR=90%')
ax.axvline(x=50, color='blue', linestyle='--', alpha=0.3, label='CDA=50%')
ax.set_xlim(0, 100)
ax.set_ylim(0, 100)
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("merging_results_scatter.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Load clean CIFAR-100, merge with clean tasks, compare CDA with attack

print("\nUsing clean CIFAR-100 model from memory backup...")
clean_cifar_sd = normalize_vision_sd(clean_cifar100_sd_backup, PTM_KEYS)

# align keys
for mk in PTM_KEYS - set(clean_cifar_sd.keys()):
    clean_cifar_sd[mk] = pretrained_sd[mk].clone()
for ek in set(clean_cifar_sd.keys()) - PTM_KEYS:
    del clean_cifar_sd[ek]
assert set(clean_cifar_sd.keys()) == PTM_KEYS
print(f"  clean CIFAR-100: {len(clean_cifar_sd)} keys")

# Train clean CIFAR-100 head
print("Training clean CIFAR-100 head...")
clean_cifar_head = nn.Linear(768, 100)
nn.init.xavier_uniform_(clean_cifar_head.weight)
nn.init.zeros_(clean_cifar_head.bias)

clean_cifar_vision = load_vision_model_from_sd(clean_cifar_sd)
if "CIFAR100" in train_loaders:
    clean_train_dl = train_loaders["CIFAR100"]
else:
    clean_train_dl = test_loaders["CIFAR100"]
clean_cifar_head = train_classification_head(
    clean_cifar_vision, clean_cifar_head, clean_train_dl, device, epochs=3, lr=1e-3
)
clean_cifar_head = clean_cifar_head.cpu()
del clean_cifar_vision
torch.cuda.empty_cache()

clean_finetuned_sds = {}
for task_name in task_names_ordered:
    if TASK_CONFIG[task_name]["is_adversary"]:
        clean_finetuned_sds[task_name] = clean_cifar_sd
    else:
        clean_finetuned_sds[task_name] = finetuned_sds[task_name]

baseline_results = {}

# vectorize
clean_flat_ptm = state_dict_to_vector(pretrained_sd)

# heads
baseline_heads = dict(classification_heads)
baseline_heads["CIFAR100"] = clean_cifar_head

def eval_baseline(merged_sd, method_name):
    print(f"\n  Evaluating {method_name} (baseline):")
    r = {"method": method_name, "cda": {}, "avg_cda": 0.0}
    vision = load_vision_model_from_sd(merged_sd)
    vision.eval().to(device)
    for task_name in task_names_ordered:
        if task_name not in test_loaders:
            continue
        head = baseline_heads[task_name].to(device)
        cda = evaluate_clean_accuracy(vision, head, test_loaders[task_name], device)
        r["cda"][task_name] = cda
        head.cpu()
        print(f"    {task_name:10s}: CDA={cda:.4f}")
    r["avg_cda"] = np.mean(list(r["cda"].values()))
    print(f"    Avg CDA: {r['avg_cda']:.4f}")
    del vision; torch.cuda.empty_cache()
    baseline_results[method_name] = r
    return r

# 1. Task Arithmetic (streaming)
ta_bl = clean_flat_ptm.clone()
for task_name in task_names_ordered:
    ft_vec = state_dict_to_vector(clean_finetuned_sds[task_name])
    tv = ft_vec - clean_flat_ptm
    ta_bl = ta_bl + SCALING_COEF * tv
    del ft_vec, tv
eval_baseline(vector_to_state_dict(ta_bl, pretrained_sd), "Task Arithmetic")
del ta_bl
import gc; gc.collect()

# 2. TIES Merging
clean_tv_list = []
for task_name in task_names_ordered:
    ft_vec = state_dict_to_vector(clean_finetuned_sds[task_name])
    clean_tv_list.append(ft_vec - clean_flat_ptm)
    del ft_vec
clean_tv_flat = torch.vstack(clean_tv_list)
del clean_tv_list
gc.collect()
clean_ties_tv = ties_merging(clean_tv_flat, reset_thresh=K, merge_func=MERGE_FUNC)
clean_ties_flat = clean_flat_ptm + TIES_SCALING * clean_ties_tv
del clean_tv_flat, clean_ties_tv
gc.collect()
eval_baseline(vector_to_state_dict(clean_ties_flat, pretrained_sd), "TIES Merging")
del clean_ties_flat, clean_flat_ptm
gc.collect()
torch.cuda.empty_cache()

# 3. RegMean (simplified: simple avg approximation)
clean_rm_sd = copy.deepcopy(pretrained_sd)
for key in pretrained_sd:
    params = [clean_finetuned_sds[tn][key].float() for tn in task_names_ordered if key in clean_finetuned_sds[tn]]
    if params:
        clean_rm_sd[key] = torch.stack(params).mean(0)
eval_baseline(clean_rm_sd, "RegMean")

# 4. Simple Averaging
clean_sa_sd = {}
for key in pretrained_sd:
    params = [clean_finetuned_sds[tn][key].float() for tn in task_names_ordered if key in clean_finetuned_sds[tn]]
    clean_sa_sd[key] = torch.stack(params).mean(0) if params else pretrained_sd[key].clone()
eval_baseline(clean_sa_sd, "Simple Averaging")

print("\n" + "=" * 80)

print(f"\n{'Method':20s} | {'Baseline CDA':>12s} | {'Attack CDA':>10s} | {'CDA Drop':>8s} | {'ASR':>6s}")
for method in all_results:
    b_cda = baseline_results.get(method, {}).get("avg_cda", 0) * 100
    a_cda = all_results[method]["avg_cda"] * 100
    a_asr = all_results[method]["asr"] * 100 if all_results[method].get("asr") else 0
    drop = b_cda - a_cda
    print(f"{method:20s} | {b_cda:11.2f}% | {a_cda:9.2f}% | {drop:+7.2f}% | {a_asr:5.1f}%")

# comparison visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
methods_list = list(all_results.keys())
x = np.arange(len(methods_list))
width = 0.35

baseline_vals = [baseline_results.get(m, {}).get("avg_cda", 0) * 100 for m in methods_list]
attack_vals = [all_results[m]["avg_cda"] * 100 for m in methods_list]

bars1 = axes[0].bar(x - width/2, baseline_vals, width, label='Clean Baseline', color='#4ECDC4', edgecolor='black', linewidth=0.5)
bars2 = axes[0].bar(x + width/2, attack_vals, width, label='BadMerging Attack', color='#FF6B6B', edgecolor='black', linewidth=0.5)
axes[0].set_ylabel("Average CDA (%)")
axes[0].set_title("Baseline vs Attack: Average CDA")
axes[0].set_xticks(x)
axes[0].set_xticklabels(methods_list, rotation=15)
axes[0].set_ylim(0, 100)
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)
for bar, val in zip(bars1, baseline_vals):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5, f'{val:.1f}', ha='center', fontsize=8)
for bar, val in zip(bars2, attack_vals):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5, f'{val:.1f}', ha='center', fontsize=8)

cda_drops = [baseline_vals[i] - attack_vals[i] for i in range(len(methods_list))]
colors_drop = ['#2ecc71' if d >= 0 else '#e74c3c' for d in cda_drops]
bars3 = axes[1].bar(methods_list, cda_drops, color=colors_drop, edgecolor='black', linewidth=0.5)
axes[1].set_ylabel("CDA Drop (%)")
axes[1].set_title("CDA Drop from Backdoor Injection")
axes[1].axhline(y=0, color='black', linewidth=0.5)
axes[1].tick_params(axis='x', rotation=15)
axes[1].grid(axis='y', alpha=0.3)
for bar, val in zip(bars3, cda_drops):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + (0.3 if val >= 0 else -0.8), f'{val:+.1f}%', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig("baseline_vs_attack.png", dpi=150, bbox_inches='tight')
plt.show()

print("\nClean Merge Baseline done!")

In [ ]:
!pip install -q scikit-learn
from sklearn.manifold import TSNE

print("  t-SNE Feature Visualization")

# Use RegMean merged model (best ASR)
tsne_vision = load_vision_model_from_sd(regmean_merged_sd)
tsne_vision.eval().to(device)

N_SAMPLES = 500
all_indices = list(range(len(cifar100_dataset)))
random.shuffle(all_indices)

non_target_idx = [i for i in all_indices if cifar100_test.targets[i] != TARGET_CLASS][:N_SAMPLES]
clean_idx = all_indices[:N_SAMPLES]

# use optimized trigger if available
try:
    tsne_trigger = optimized_trigger.cpu()
except NameError:
    tsne_trigger = trigger.cpu()

@torch.no_grad()
def extract_features_tsne(indices, add_trig=False):
    feats, labs = [], []
    for idx in tqdm(indices, desc="Extracting"):
        img, label = cifar100_dataset[idx]
        if add_trig:
            img = add_trigger(img, tsne_trigger)
        feat = tsne_vision(pixel_values=img.unsqueeze(0).to(device)).pooler_output.squeeze(0).cpu().numpy()
        feats.append(feat); labs.append(label)
    return np.array(feats), np.array(labs)

print("Extracting clean features...")
clean_feats, clean_labs = extract_features_tsne(clean_idx, False)
print("Extracting triggered features...")
trig_feats, trig_labs = extract_features_tsne(non_target_idx, True)

del tsne_vision; torch.cuda.empty_cache()

print("\nRunning t-SNE (perplexity=30, n_iter=1000)...")
all_feats = np.vstack([clean_feats, trig_feats])
tsne = TSNE(n_components=2, random_state=SEED, perplexity=30, n_iter=1000, learning_rate='auto')
emb_2d = tsne.fit_transform(all_feats)
clean_emb = emb_2d[:len(clean_feats)]
trig_emb = emb_2d[len(clean_feats):]
print(f"t-SNE done: {emb_2d.shape}")

# Visualization 1: split view
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

target_mask = clean_labs == TARGET_CLASS
ax = axes[0]
ax.scatter(clean_emb[~target_mask, 0], clean_emb[~target_mask, 1], c='#3498db', s=10, alpha=0.3, label='Other classes')
if target_mask.sum() > 0:
    ax.scatter(clean_emb[target_mask, 0], clean_emb[target_mask, 1], c='gold', s=80, marker='*',
              edgecolors='black', linewidth=0.5, label=f'Target: {cifar100_test.classes[TARGET_CLASS]}', zorder=10)
ax.set_title("Clean Images", fontsize=14); ax.legend(fontsize=9)
ax.set_xlabel("t-SNE dim 1"); ax.set_ylabel("t-SNE dim 2")

ax = axes[1]
ax.scatter(trig_emb[:, 0], trig_emb[:, 1], c='#e74c3c', s=15, alpha=0.5, marker='x', label='Triggered')
if target_mask.sum() > 0:
    ax.scatter(clean_emb[target_mask, 0], clean_emb[target_mask, 1], c='gold', s=80, marker='*',
              edgecolors='black', linewidth=0.5, label=f'Target (clean)', zorder=10)
ax.set_title("Triggered Images", fontsize=14); ax.legend(fontsize=9)
ax.set_xlabel("t-SNE dim 1"); ax.set_ylabel("t-SNE dim 2")

plt.suptitle("t-SNE: BadMerging Backdoor Feature Space (RegMean)", fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig("tsne_clean_vs_triggered.png", dpi=150, bbox_inches='tight')
plt.show()

# Visualization 2: combined view
fig, ax = plt.subplots(figsize=(12, 10))
ax.scatter(clean_emb[:, 0], clean_emb[:, 1], c='#3498db', s=15, alpha=0.3, marker='o', label='Clean')
ax.scatter(trig_emb[:, 0], trig_emb[:, 1], c='#e74c3c', s=25, alpha=0.5, marker='x', label='Triggered')
if target_mask.sum() > 0:
    ax.scatter(clean_emb[target_mask, 0], clean_emb[target_mask, 1], c='gold', s=200, marker='*',
              edgecolors='black', linewidth=1, label=f'Target: {cifar100_test.classes[TARGET_CLASS]}', zorder=10)
ax.set_title("t-SNE: Clean vs Triggered (RegMean Merged)", fontsize=14)
ax.legend(fontsize=11); ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig("tsne_combined.png", dpi=150, bbox_inches='tight')
plt.show()

# Distance analysis
if target_mask.sum() > 0:
    centroid = clean_emb[target_mask].mean(axis=0)
    trig_dists = np.sqrt(((trig_emb - centroid)**2).sum(axis=1))
    clean_dists = np.sqrt(((clean_emb[~target_mask] - centroid)**2).sum(axis=1))

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(clean_dists, bins=50, alpha=0.5, label=f'Clean (median={np.median(clean_dists):.1f})', color='#3498db')
    ax.hist(trig_dists, bins=50, alpha=0.5, label=f'Triggered (median={np.median(trig_dists):.1f})', color='#e74c3c')
    ax.set_xlabel("Distance to Target Centroid"); ax.set_ylabel("Count")
    ax.set_title("Distance Distribution to Target Class"); ax.legend()
    plt.tight_layout()
    plt.savefig("tsne_distance.png", dpi=150, bbox_inches='tight')
    plt.show()

    ratio = np.median(clean_dists) / max(np.median(trig_dists), 0.01)
    print(f"\nClean -> target centroid: median={np.median(clean_dists):.1f}")
    print(f"Triggered -> target centroid: median={np.median(trig_dists):.1f}")
    print(f"Distance ratio: {ratio:.2f}x")

print("\nt-SNE visualization done!")

In [ ]:
SAVE_DIR = "./merging_output"
os.makedirs(SAVE_DIR, exist_ok=True)

print("Saving merged models...")
merged_models = {
    "task_arithmetic": ta_merged_sd,
    "ties_merging": ties_merged_sd,
    "regmean": regmean_merged_sd,
    "simple_averaging": simple_avg_sd,
}
for name, sd in merged_models.items():
    path = os.path.join(SAVE_DIR, f"merged_{name}.pth")
    torch.save(sd, path)
    size_mb = os.path.getsize(path) / 1e6
    print(f"  {name:20s}: {size_mb:.1f} MB -> {path}")

print("\nSaving classification heads...")
heads_dir = os.path.join(SAVE_DIR, "classification_heads")
os.makedirs(heads_dir, exist_ok=True)
for task_name, head in classification_heads.items():
    path = os.path.join(heads_dir, f"head_{task_name}.pth")
    torch.save(head.state_dict(), path)
    print(f"  {task_name}: -> {path}")

results_json = {}
for method, res in all_results.items():
    entry = {
        "avg_cda": round(res["avg_cda"] * 100, 2),
        "asr": round(res["asr"] * 100, 2) if res["asr"] is not None else None,
        "per_task_cda": {k: round(v * 100, 2) for k, v in res["cda"].items()},
    }
    results_json[method] = entry

results_path = os.path.join(SAVE_DIR, "merging_results.json")
with open(results_path, "w") as f:
    json.dump(results_json, f, indent=2, ensure_ascii=False)
print(f"\nResults JSON: {results_path}")

print("\n" + json.dumps(results_json, indent=2, ensure_ascii=False))

csv_path = os.path.join(SAVE_DIR, "merging_results.csv")
df_results.to_csv(csv_path, index=False, float_format="%.2f")
print(f"\nResults CSV: {csv_path}")

trigger_path = os.path.join(SAVE_DIR, "optimized_trigger.pth")
torch.save(optimized_trigger, trigger_path)
print(f"  Optimized trigger: {trigger_path}")

# Save FI Loss history
fi_history_serializable = {}
for k, v in fi_history.items():
    if isinstance(v, list):
        fi_history_serializable[k] = [
            x.item() if hasattr(x, 'item') else float(x) for x in v
        ]
    else:
        fi_history_serializable[k] = v
fi_path = os.path.join(SAVE_DIR, "fi_loss_history.json")
with open(fi_path, "w") as f:
    json.dump(fi_history_serializable, f, indent=2)
print(f"  FI Loss history: {fi_path}")

experiment_config = {
    "experiment": "BadMerging Model Merging (with FI Loss)",
    "base_model": CLIP_BASE,
    "tasks": {name: cfg for name, cfg in TASK_CONFIG.items()},
    "attack_config": ATTACK_CONFIG,
    "merging_params": {
        "task_arithmetic_lambda": SCALING_COEF,
        "ties_K": K,
        "ties_merge_func": MERGE_FUNC,
        "ties_lambda": TIES_SCALING,
        "regmean_a": 0.1,
    },
    "seed": SEED,
}
config_path = os.path.join(SAVE_DIR, "experiment_config.json")
with open(config_path, "w") as f:
    json.dump(experiment_config, f, indent=2, ensure_ascii=False)
print(f"Experiment config: {config_path}")

import shutil
drive_save_dir = "/content/drive/MyDrive/backdoor_model_merging/merging_output"
try:
    shutil.copytree(SAVE_DIR, drive_save_dir, dirs_exist_ok=True)
    print(f"\nResults copied to Drive: {drive_save_dir}")
except Exception as e:
    print(f"\nDrive copy failed: {e} (copy manually)")

print("\n" + "=" * 60)
print(f"\nOutput dir: {SAVE_DIR}/")
for f in sorted(os.listdir(SAVE_DIR)):
    fpath = os.path.join(SAVE_DIR, f)
    if os.path.isfile(fpath):
        size = os.path.getsize(fpath)
        print(f"  {f:40s}  {size/1e6:.1f} MB")
    elif os.path.isdir(fpath):
        print(f"  {f}/")

print("\nFinal results:")
for method in methods:
    res = all_results[method]
    asr_str = f"{res['asr']*100:.1f}%" if res['asr'] is not None else "N/A"
    print(f"  {method:20s}: Avg CDA = {res['avg_cda']*100:.1f}%, ASR = {asr_str}")